# Traceprop-LLM -- GPU evidence run: LogIX at matched storage (speed + LDS + reverse match), item 9 rerun

Run order (priority order from the last review round -- item 1 decides the paper's core novelty framing, so it goes first):

1. **LogIX at matched storage.** Shrink LogIX's rank to Traceprop's ~2KB/example budget (exp31, already run once for speed) and now ALSO measure LDS at that same rank (exp35, new). Then run the **reverse match**: leave LogIX at its own default rank (~96KB/example on GPT-2's tracked scope) and grow Traceprop's `proj_dim` up to that same budget instead -- two matched comparison points, harder to dispute than one.
2. **Item 9 rerun.** exp29 (inline vs. final-checkpoint LDS) with the raw per-test-example npz saved, so the paired bootstrap can run locally afterward without another GPU session.
3. **exp34** (Table 2 + tracked-parameters sweep) -- lowest priority this session; skip if you're already running it separately or out of time.

Every script now refuses to overwrite an existing output file (`--out`/`--force`) and asserts the GPU is an L4 (`--gpu_check`/`--skip_gpu_check`) before running -- if a run gets interrupted and rerun, it will not silently clobber a result or silently mix in a different GPU.

## Setup: pin versions, mount Drive, assert L4, clone repo

In [ ]:
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
assert 'L4' in torch.cuda.get_device_name(0), (
    f"expected an L4, got {torch.cuda.get_device_name(0)} -- Table 1/2, the sweep, and this "
    f"session's runs are all L4-only; a different GPU would make the numbers incomparable. "
    f"Reconnect and request an L4 runtime."
)

# Pin to known-good versions -- an unpinned `transformers` install can pull a
# broken bleeding-edge release; numpy>=2 breaks the torch build Colab ships.
!pip -q install "transformers==4.44.2" "peft==0.13.2" "accelerate==0.34.2" "numpy<2" datasets scipy
!pip -q install --ignore-requires-python logix-ai

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/traceprop_runs', exist_ok=True)

In [ ]:
import getpass
TOKEN = getpass.getpass('GitHub token: ').strip()
url = f'https://{TOKEN}@github.com/AmitoVrito/Traceprop.git'
!git clone -q {url} /content/Traceprop || (cd /content/Traceprop && git pull -q)
%cd /content/Traceprop
!pip -q install -e .
%cd /content/Traceprop/experiments

## Step 1a: LogIX at matched storage -- speed (exp31, both rank_modes)

`rank_mode=matched` shrinks LogIX down to our ~2KB/example budget (this is the number already drafted into Table 3). `rank_mode=logix_default` leaves LogIX at its own default rank and prints/saves the `proj_dim` Traceprop would need to grow up to LogIX's ~96KB/example budget -- read off `storage_matching.reverse_match_proj_dim` from its output JSON in the next cell before running Step 1c.

In [ ]:
!python exp31_logix_comparison.py --backend hf --model gpt2 --device cuda \
    --steps 200 --track 1 --proj_dim 512 --rank_mode matched \
    --out results/exp31_logix_hf_gpt2_matched.json --force
!python exp31_logix_comparison.py --backend hf --model gpt2 --device cuda \
    --steps 200 --track 1 --proj_dim 512 --rank_mode logix_default \
    --out results/exp31_logix_hf_gpt2_logix_default.json --force
!cp results/exp31_logix_hf_gpt2_matched.json results/exp31_logix_hf_gpt2_logix_default.json \
    /content/drive/MyDrive/traceprop_runs/ 2>/dev/null

In [ ]:
import json
d = json.load(open('results/exp31_logix_hf_gpt2_logix_default.json'))
reverse_proj_dim = d['storage_matching']['reverse_match_proj_dim']
print('reverse_match_proj_dim =', reverse_proj_dim)
print('logix_default overhead (matched_buffering):', d['configs']['matched_buffering']['overhead_pct_median'], '%')
d2 = json.load(open('results/exp31_logix_hf_gpt2_matched.json'))
print('matched overhead (matched_buffering):', d2['configs']['matched_buffering']['overhead_pct_median'], '%')

## Step 1b: LogIX at matched storage -- LDS quality (exp35, new)

Trains one GPT-2/SST-2 target model, scores Traceprop-dot/trak and LogIX-dot/preconditioned (via LogIX's own `compute_influence_all`) against the same subset-retraining ground truth. `n_subsets=200` here is lower-powered than the 500 used for the paper's canonical LDS numbers (exp29) -- this is a new comparison, not a replacement for that one -- raise it if there's time budget left. Rough cost: dominated by the subset-retraining loop, same order of magnitude per-subset as exp29's item-9 run.

In [ ]:
!python exp35_logix_lds.py --backend hf --data sst2 --model gpt2 --device cuda \
    --n_train 2000 --n_test 200 --n_subsets 200 --subset_frac 0.5 --epochs 3 \
    --batch 16 --proj_dim 512 --track 1 --rank_mode matched \
    --out results/exp35_logix_lds_gpt2_matched.json --force
!cp results/exp35_logix_lds_gpt2_matched.json results/exp35_logix_lds_gpt2_matched_raw.npz \
    /content/drive/MyDrive/traceprop_runs/ 2>/dev/null

## Step 1c: reverse match -- LogIX at its own default rank (LDS), and Traceprop grown to that same budget

Two calls: exp35 at `rank_mode=logix_default` (LogIX's own default rank, no shrinking) gives LogIX's LDS at its native ~96KB/example budget. Then exp29 at `--proj_dim {reverse_proj_dim}` gives Traceprop's own LDS grown UP to that same budget -- the second matched comparison point.

In [ ]:
!python exp35_logix_lds.py --backend hf --data sst2 --model gpt2 --device cuda \
    --n_train 2000 --n_test 200 --n_subsets 200 --subset_frac 0.5 --epochs 3 \
    --batch 16 --proj_dim 512 --track 1 --rank_mode logix_default \
    --out results/exp35_logix_lds_gpt2_logix_default.json --force
!cp results/exp35_logix_lds_gpt2_logix_default.json results/exp35_logix_lds_gpt2_logix_default_raw.npz \
    /content/drive/MyDrive/traceprop_runs/ 2>/dev/null

In [ ]:
# Traceprop grown up to LogIX's default budget (reverse_proj_dim from Step 1a).
# Reuses exp29's harness (final-checkpoint dot/trak LDS) at the larger proj_dim --
# same 200-subset ground truth size as Step 1b/1c so all four numbers are comparable.
!python exp29_inline_vs_final_lds.py --backend hf --data sst2 --model gpt2 --device cuda \
    --n_train 2000 --n_test 200 --n_subsets 200 --subset_frac 0.5 --epochs 3 \
    --batch 16 --proj_dim {reverse_proj_dim} --track 1 \
    --out results/exp29_hf_gpt2_reverse_match.json --force
!cp results/exp29_hf_gpt2_reverse_match.json results/exp29_hf_gpt2_reverse_match_raw.npz \
    /content/drive/MyDrive/traceprop_runs/ 2>/dev/null

## Step 1 result: read everything back

Four LDS numbers to compare pairwise: Traceprop @ 2KB vs. LogIX @ 2KB (matched point), and LogIX @ 96KB vs. Traceprop @ 96KB (reverse-match point). Bring this table back to Claude Code -- it drafts the §3.3 text for whichever of the three outcomes (Traceprop still wins / roughly equal / LogIX wins) this data actually shows.

In [ ]:
import json
print('=== Matched budget (~2KB/example) ===')
d = json.load(open('results/exp35_logix_lds_gpt2_matched.json'))
for k, v in d['lds'].items():
    print(f"  {k:<22} {v['mean']:+.4f} +/- {v['std']:.4f}")
print('  logix_rank_used =', d['storage_matching']['logix_rank_used'])

print('\n=== Reverse-match budget (LogIX default, ~96KB/example) ===')
d = json.load(open('results/exp35_logix_lds_gpt2_logix_default.json'))
for k, v in d['lds'].items():
    print(f"  {k:<22} {v['mean']:+.4f} +/- {v['std']:.4f}")
print('  logix_rank_used =', d['storage_matching']['logix_rank_used'],
      ' reverse_match_proj_dim =', d['storage_matching']['reverse_match_proj_dim'])

print('\n=== Traceprop grown to LogIX default budget (exp29 reverse match) ===')
d = json.load(open('results/exp29_hf_gpt2_reverse_match.json'))
for k, v in d['lds'].items():
    print(f"  {k:<22} {v['mean']:+.4f} +/- {v['std']:.4f}")
print('  proj_dim =', d['proj_dim'])

## Step 2: item 9 rerun -- inline vs. final-checkpoint LDS, real repeats, raw npz saved

This is the number already in the paper's §3.4 (0.0871+/-0.0489 dot / 0.1735+/-0.0661 TRAK inline vs. 0.0314+/-0.0668 / 0.0970+/-0.0507 final). exp29's current code already saves the raw per-test-example Spearman-r arrays plus the masks/margins matrices to `*_raw.npz` -- that npz is what the paired bootstrap needs and didn't exist for the version already in the paper. Rerun at the same settings so the paper's headline numbers get bootstrap-ready data; the new run's mean/std should land close to the existing numbers (same seed, same everything) as a consistency check.

In [ ]:
!python exp29_inline_vs_final_lds.py --backend hf --data sst2 --model gpt2 --device cuda \
    --n_train 2000 --n_test 200 --n_subsets 500 --subset_frac 0.5 --epochs 3 \
    --batch 16 --proj_dim 512 --track 1 \
    --out results/exp29_hf_gpt2_rerun.json --force
!cp results/exp29_hf_gpt2_rerun.json results/exp29_hf_gpt2_rerun_raw.npz \
    /content/drive/MyDrive/traceprop_runs/ 2>/dev/null

In [ ]:
import json
d = json.load(open('results/exp29_hf_gpt2_rerun.json'))
for k, v in d['lds'].items():
    print(f"  {k:<14} {v['mean']:+.4f} +/- {v['std']:.4f}")

## Step 3 (lowest priority, optional): exp34 -- Table 2 + tracked-parameters sweep

Only run this if Steps 1-2 finished with time to spare, or if it isn't already running in a separate session. Single consolidated script (no per-cell state to go stale), resumable -- reruns skip already-completed jobs.

In [ ]:
!python exp34_table2_and_sweep.py --drive_dir /content/drive/MyDrive/traceprop_runs

## Bring results back

Paste the printed JSON/tables from Step 1's result cell (and Step 2's, and Step 3's if run) back to Claude Code. It will draft/update §3.3 (LogIX comparison) with whichever of the three outcomes the matched + reverse-match LDS numbers actually show, fold the item-9 rerun into §3.4, and update Table 2 / the sweep paragraph if Step 3 ran.